<a href="https://colab.research.google.com/github/GreatLakesCommission/IEDRR_inland_lakes/blob/main/get_spp_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''
do not run - copy to page console to prevent accidental disconnections


function ClickConnect() {
  console.log('Working')
  document
    .querySelector('#top-toolbar > colab-connect-button')
    .shadowRoot.querySelector('#connect')
    .click()
}
intervalTiming = setInterval(ClickConnect, 60000)

'''

In [1]:
# general setup
%%capture
from google.colab import drive
drive.mount('/content/drive')
import datetime
import os
os.chdir("drive/My Drive/iedrr")
today = datetime.date.today().strftime('%Y%m%d')
outfolder = "speciesobs_"+today
if not os.path.exists(outfolder):
  os.mkdir(outfolder)
os.chdir(outfolder)
import pandas as pd
!pip install pyreadr
import pyreadr
from scipy import spatial
from google.colab import userdata



In [2]:
# invasive species of interest, include alternate scientific names where more than one is in use


fishlist = {
    'sci_name': ['Channa', 'Clarias batrachus', 'Gymnocephalus cernua', 'Misgurnus anguillicaudatus', 'Neogobius melanostomus', 'Osmerus mordax', 'Osteoglossum bicirrhosum', 'Proterorhinus semilunaris', 'Tinca tinca'],
    'common_name': ['Snakeheads', 'Walking catfish', 'Ruffe', 'Pond loach', 'Round goby', 'Rainbow smelt', 'Silver arowana', 'Tubenose goby', 'Tench']
}
fishlist = pd.DataFrame(data=fishlist)
fishlist['taxon'] = 'fish'


plantlist = {
    'sci_name': ['Alternanthera philoxeroides', 'Cabomba caroliniana', 'Callitriche stagnalis', 'Elodea densa', 'Hottonia palustris', 'Hydrilla verticillata', 'Hydrocharis morsus-ranae', 'Hygrophila polysperma', 'Limnophila sessiliflora', 'Ludwigia grandiflora', 'Ludwigia hexapetala', 'Ludwigia peploides', 'Marsilea mutica', 'Marsilea quadrifolia', 'Myriophyllum aquaticum', 'Najas minor', 'Nasturtium officinale', 'Nelumbo nucifera', 'Nitellopsis obtusa', 'Ottelia alismoides', 'Pistia stratiotes', 'Pontederia azurea', 'Pontederia crassipes', 'Sagittaria sagittifolia', 'Salvinia auriculata', 'Salvinia molesta', 'Spirodela punctata', 'Stratiotes aloides', 'Trapa natans'],
    'common_name': ['Alligator weed', 'Carolina fanwort', 'Pond water-starwort', 'Brazilian waterweed', 'Water violet', 'Hydrilla', 'European frog-bit', 'Indian swampweed', 'Dwarf ambulia', 'Large-flower primrose-willow', 'Six petal water primrose', 'Creeping water primrose', 'Australian water-clover', 'European water-clover', 'Parrot feather', 'Brittle naiad', 'Water-cress', 'Sacred lotus', 'Starry stonewort', 'Duck-lettuce', 'Water lettuce', 'Anchored water-hyacinth', 'Common water-hyacinth', 'Hawaii arrowhead', 'Eared salvinia', 'Giant salvinia', 'Dotted duckweed', 'Water soldier', 'European water chestnut']
}

# plantlist_slim doesn't include emergent species
plantlist_slim = {
    'sci_name': ['Cabomba caroliniana', 'Callitriche stagnalis', 'Elodea densa', 'Hydrilla verticillata', 'Hydrocharis morsus-ranae', 'Hygrophila polysperma', 'Limnophila sessiliflora', 'Marsilea mutica', 'Marsilea quadrifolia', 'Myriophyllum aquaticum', 'Najas minor', 'Nelumbo nucifera', 'Nitellopsis obtusa', 'Ottelia alismoides', 'Pistia stratiotes', 'Pontederia azurea', 'Pontederia crassipes', 'Salvinia auriculata', 'Salvinia molesta', 'Spirodela punctata', 'Stratiotes aloides', 'Trapa natans'],
    'common_name': ['Carolina fanwort', 'Pond water-starwort', 'Brazilian waterweed', 'Hydrilla', 'European frog-bit', 'Indian swampweed', 'Dwarf ambulia', 'Australian water-clover', 'European water-clover', 'Parrot feather', 'Brittle naiad', 'Sacred lotus', 'Starry stonewort', 'Duck-lettuce', 'Water lettuce', 'Anchored water-hyacinth', 'Common water-hyacinth', 'Eared salvinia', 'Giant salvinia', 'Dotted duckweed', 'Water soldier', 'European water chestnut']
}

plantlist = pd.DataFrame(data=plantlist)
plantlist['taxon'] = 'plant'


invertlist = {
    'sci_name': ['Bithynia tentaculata', 'Bythotrephes longimanus', 'Cercopagis pengoi', 'Corbicula fluminea', 'Dreissena bugensis', 'Dreissena polymorpha', 'Eriocheir sinensis', 'Hemimysis anomala', 'Melanoides tuberculata', 'Potamopyrgus antipodarum', 'Procambarus virginalis'],
    'common_name': ['Faucet snail', 'Spiny water flea', 'Fishhook waterflea', 'Basket clam', 'Quagga mussel', 'Zebra mussel', 'Mitten crab', 'Bloody red shrimp', 'Red-rimmed melania', 'New Zealand mud snail', 'Marbled crayfish (Marmorkrebs)']
}

invertlist = pd.DataFrame(data=invertlist)
invertlist['taxon'] = 'invertebrate'

my_vars = {}

my_vars["fish"] = fishlist
my_vars["plant"] = plantlist
my_vars["invert"] = invertlist


# get institution lat/longs for QAQC
url = "https://github.com/ropensci/CoordinateCleaner/raw/refs/heads/master/data/institutions.rda"
dst_path = os.path.join(os.getcwd(), "institutions.rda") # download to CoLab
res = pyreadr.read_r(pyreadr.download_file(url, dst_path)) # convert rda to dictionary of dataframes

institutions = res["institutions"]
# filter to institutions within GL bounding box
institutions = institutions[(institutions['decimalLatitude'].between(36.9171,49.6117)) & (institutions['decimalLongitude'].between(-100.5513,-71.79))]
# set up a k-dimensional tree
inst_coords = list(zip(institutions["decimalLatitude"], institutions["decimalLongitude"]))
tree = spatial.KDTree(inst_coords)

def calculate_min(row):
    return tree.query([(row["decimalLatitude"],row["decimalLongitude"])])[0][0]

!rm institutions.rda

GBIF

In [9]:
#GBIF setup
%%capture
!pip install pygbif
# capture suppresses output

#setup
from pygbif import species as species
from pygbif import occurrences as occ
from pygbif.occurrences.download import GbifDownload
import os
import glob
import datetime
from time import sleep
import zipfile
# add your gbif.org username, password and contact email for download notices to Colab's 'Secrets'
# toggle notebook access on for all three
%env GBIF_USER=userdata.get('GBIF_USER')
%env GBIF_PWD = userdata.get('GBIF_PWD')
%env GBIF_EMAIL = userdata.get('GBIF_EMAIL')



SLEEP_DURATION = 20


In [23]:
# download GBIF species obs since 1970 within GL bounding box as zip files via API


gbif_obs = {}

skip = [] #skip taxa you previously exported a csv for

# get GBIF taxon IDs based on scientific names
def getskey(z):
  return species.name_backbone(z)['usageKey']

for i, (k, v) in enumerate(my_vars.items()):
  if not k in skip:
    records = []
    print("downloading ", k)
    splist = v['sci_name'].tolist()
    splist = list(set(splist))
    spkeys = [ getskey(x) for x in splist ]
    spkeys = list(map(str, spkeys))
    #res = occurrences.download(*args, **kwargs, hasCoordinate=True, hasGeospatialIssue=False, decimalLatitude='36.9171,49.6117', decimalLongitude='-100.5513,-71.79', year="1970,2025", pred_type='and', basisOfRecord = ['HUMAN_OBSERVATION', 'OBSERVATION', 'MACHINE_OBSERVATION', 'LIVING_SPECIMEN', 'MATERIAL_SAMPLE'])

    #res = paginated_search(500000, key=skey, data='children')
    # construct query
    gbif_query = GbifDownload(userdata.get('GBIF_USER'), userdata.get('GBIF_EMAIL'))
    gbif_query.add_predicate_dict({"type": "in", "key": "BASIS_OF_RECORD", "values": ['HUMAN_OBSERVATION', 'OBSERVATION', 'MACHINE_OBSERVATION', 'LIVING_SPECIMEN', 'MATERIAL_SAMPLE'], "matchCase": "false"})
    gbif_query.add_predicate_dict({"type": "equals", "key": 'HAS_COORDINATE', 'value': 'TRUE', "matchCase": "false"})
    gbif_query.add_predicate_dict({"type": "equals", "key": 'HAS_GEOSPATIAL_ISSUE', 'value': 'FALSE', "matchCase": "false"})
    gbif_query.add_predicate_dict({"type": "within", "geometry": "POLYGON((-100.551 36.917,-71.79 36.917,-71.79 49.612,-100.551 49.612,-100.551 36.917))"})
    gbif_query.add_predicate_dict({"type": "greaterThanOrEquals", "key": 'YEAR', 'value': '1970', "matchCase": "false"})
    gbif_query.add_predicate_dict({"type": "in", "key": 'TAXON_KEY', 'values': spkeys, "matchCase": "false"})
    # submit download query
    xx = gbif_query.post_download(userdata.get('GBIF_USER'), userdata.get('GBIF_PWD'))
    # wait for download to be ready
    while True:
      print(f"waiting to get download {xx}...")
      status = occ.download_meta(key = xx)['status']

      if status not in ['PREPARING', 'RUNNING']:  # = not ready yet
          if status == 'SUCCEEDED':
              print(f"Download is ready, getting it")
              output_path = k+"_gbif_obs"
              if os.path.exists(output_path): # get rid of any previous downloads for this run
                files = glob.glob(output_path+'/*.zip')
                for f in files:
                  os.remove(f)
              else:
                os.mkdir(output_path)

              occ.download_get(xx, output_path)
          else:
              print(f"Status is {status}, why?")
              print(occ.download_meta(key = xx))
          break

      sleep(SLEEP_DURATION)

    print("finished with ", k)


downloading  fish
waiting to get download 0071788-241126133413365...
Download is ready, getting it
finished with  fish
downloading  plant
waiting to get download 0071795-241126133413365...
Download is ready, getting it
finished with  plant
downloading  invert
waiting to get download 0071800-241126133413365...
Download is ready, getting it
finished with  invert


In [34]:
    # read the GBIF data back in and QA/QC it
    for k in ['fish', 'plant', 'invert']:
      # read zip file into dataframe
      output_path = k+"_gbif_obs"
      files = os.listdir(output_path)
      file_path = os.path.join(output_path, files[0])
      print(file_path)
      base_name, extension = os.path.splitext(files[0])
      zf = zipfile.ZipFile(file_path)
      df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')

      print(len(df.index), k, " records")

      #drop observations whose coordinates are the location of an institution
      df["mindist"] = df.apply(calculate_min, axis=1)


      df = df[df["mindist"] > 0.00000000001] # buffer in degrees

      # drop duplicate locations within each species
      df = df.groupby('taxonKey').apply(lambda x: x.drop_duplicates(['decimalLatitude', 'decimalLongitude']))

      # export to csv before continuing because this cell will take forever
      df.to_csv(output_path + '/'+ k +'_obs_gbifraw_' + datetime.date.today().strftime('%Y%m%d') + '.csv', index=False)
      # add to dict
      gbif_obs[k] = df



fish_gbif_obs/0071788-241126133413365.zip


<ipython-input-34-436bc3795b9a>:10: DtypeWarning: Columns (17,29,36,37,38,39,40,41,43,44,46,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')


49881 fish  records


<ipython-input-34-436bc3795b9a>:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('taxonKey').apply(lambda x: x.drop_duplicates(['decimalLatitude', 'decimalLongitude']))


plant_gbif_obs/0071795-241126133413365.zip


<ipython-input-34-436bc3795b9a>:10: DtypeWarning: Columns (39,46) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')


19345 plant  records


<ipython-input-34-436bc3795b9a>:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('taxonKey').apply(lambda x: x.drop_duplicates(['decimalLatitude', 'decimalLongitude']))


invert_gbif_obs/0071800-241126133413365.zip
8874 invert  records


<ipython-input-34-436bc3795b9a>:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('taxonKey').apply(lambda x: x.drop_duplicates(['decimalLatitude', 'decimalLongitude']))


GLANSIS

In [38]:
import requests

url = "https://nas.er.usgs.gov/ipt/archive.do?r=nas_glansis"
filename = "GLANSIS_{}.zip".format(today)  # Choose a name for the downloaded file

response = requests.get(url)

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print("Zip file downloaded successfully.")
else:
    print("Failed to download the zip file.")

Zip file downloaded successfully.


In [56]:
zf = zipfile.ZipFile(filename)
df = pd.read_csv(zf.open('occurrence.txt'), sep='\t')

df['eventDate'] = pd.to_datetime(df['eventDate'], format="%Y-%m-%d", errors='coerce')
# Drop rows with invalid dates
df = df.dropna(subset='eventDate')

#QAQC
df = df.loc[(df['decimalLatitude'] >= 36.917) & (df['decimalLatitude'] <= 49.612) & (df['decimalLongitude'] >= -100.551) & (df['decimalLongitude'] <= -71.79)] # bounding box
df = df.loc[df['eventDate']>datetime.datetime(1970,1,1)] # drop old data
df = df.loc[df['georeferenceRemarks'] != "Centroid"] # drop obs with poor coordinates

glansis_obs = {}


for i, (k, v) in enumerate(my_vars.items()):
  splist = v['sci_name'].tolist()
  splist = list(set(splist))
  if "Elodea densa" in splist:
    splist.append("Egeria densa") # GLANSIS is using Egeria densa instead of Elodea densa
  hightax = []
  for item in splist:
    if len(item.split()) == 1:
      hightax.append(item)

  # drop nontarget species
  ndf = df[(df["scientificName"].isin(splist)) | (df["genus"].isin(hightax))]

  glansis_obs[k] = ndf

MISIN

In [11]:
%%capture
!pip install esri2gpd
import esri2gpd

# MISIN observations layer updated daily
url = "https://services.arcgis.com/uHAHKfH1Z5ye1Oe0/arcgis/rest/services/misin_database_obs/FeatureServer/0"

misin_obs = {}

# convert esri date to datetime
def convert_esri_date(row):
    """Converts an esriFieldTypeDate value to a Python datetime object."""
    return datetime.datetime.fromtimestamp(row["DAY"] / 1000)  # Divide by 1000 to get seconds

for i, (k, v) in enumerate(my_vars.items()):
  # separate list of genera
  splist = v['sci_name'].tolist()
  splist = list(set(splist))
  # separate out genera with no species epithet
  hightax = []
  for item in splist:
    if len(item.split()) == 1:
      hightax.append(item)
  if "Elodea densa" in splist:
    splist.append("Egeria densa") # MISIN is using Egeria densa instead of Elodea densa
  genus = [x.split()[0] for x in splist]

  gdf = esri2gpd.get(url, fields=['RECORDID', 'OBSERVER', 'DAY', 'LATITUDE', 'LONGITUDE', 'GENUS', 'SPECIES', 'VERIFIED'], where=f"GENUS IN {tuple(genus)}")
  gdf["DAY"] = gdf.apply(convert_esri_date, axis=1)
  #QAQC
  gdf = gdf[gdf["VERIFIED"] == 2]  # for field VERIFIED, 2 = 'Trusted Source', only keep these
  gdf = gdf.loc[(gdf['LATITUDE'] >= 36.917) & (gdf['LATITUDE'] <= 49.612) & (gdf['LONGITUDE'] >= -100.551) & (gdf['LONGITUDE'] <= -71.79)] # bounding box
  gdf = gdf.loc[gdf['DAY']>datetime.datetime(1970,1,1)] # drop old data
  # drop nontarget species
  gdf["sci_name"] = gdf["GENUS"] + " " + gdf["SPECIES"]
  gdf = gdf[(gdf["sci_name"].isin(splist)) | (gdf["GENUS"].isin(hightax))]

  misin_obs[k] = gdf

iMapInvasives

In [22]:
%%capture
!pip install esri2gpd
import esri2gpd
# in GL states, iMapInvasives obs are almost exclusively in PA and NY

url = "https://imapinvasives.natureserve.org/arcgis/rest/services/public_presence/MapServer/4" # presences
url2 = "https://imapinvasives.natureserve.org/arcgis/rest/services/public_approximate_presence/MapServer/4" # approx coordinates presences

imap_obs = {}

import geopandas as gpd

In [25]:

# convert esri date to datetime
def convert_esri_date(row):
    """Converts an esriFieldTypeDate value to a Python datetime object."""
    return datetime.datetime.fromtimestamp(row["observation_date"] / 1000)  # Divide by 1000 to get seconds

for i, (k, v) in enumerate(my_vars.items()):
  # separate list of genera
  splist = v['sci_name'].tolist()
  splist = list(set(splist))
  # separate out genera with no species epithet
  hightax = []
  for item in splist:
    if len(item.split()) == 1:
      hightax.append(item)
  if "Elodea densa" in splist:
    splist.append("Egeria densa") # iMapInvasives is using Egeria densa instead of Elodea densa
  genus = [x.split()[0] for x in splist]

  gdf = esri2gpd.get(url, fields=['present_species_id', 'observer_name', 'observation_date', 'jurisdiction', 'genus', 'scientific_name'], where=f"genus IN {tuple(genus)} AND jurisdiction IN ('New York', 'Pennsylvania', 'Michigan', 'Ohio')")
  gdf2 = esri2gpd.get(url2, fields=['present_species_id', 'observer_name', 'observation_date', 'jurisdiction', 'genus', 'scientific_name'], where=f"genus IN {tuple(genus)} AND jurisdiction IN ('New York', 'Pennsylvania', 'Michigan', 'Ohio')")
  rdf = gpd.GeoDataFrame(pd.concat([gdf, gdf2], ignore_index=True), crs=gdf.crs)

  rdf["observation_date"] = rdf.apply(convert_esri_date, axis=1)
  #QAQC
  rdf = rdf.loc[rdf['observation_date']>datetime.datetime(1970,1,1)] # drop old data
  # drop nontarget species
  rdf = rdf[(rdf["scientific_name"].isin(splist)) | (rdf["genus"].isin(hightax))]
  # all obs in layer have been confirmed

  imap_obs[k] = rdf


/usr/local/lib/python3.10/dist-packages/esri2gpd/core.py:91: UserWarning: Long download time — total download will require 60 separate requests
  warnings.warn(


EDDMapS

In [3]:
from google.colab import userdata
import requests
import json
from pandas import json_normalize

In [63]:
# log in to EDDMapS API
s = requests.session()

login_url = 'https://api.bugwoodcloud.org/v2/login'
headers = {
    "accept": "application/json",
    "Content-Type": "application/json"
}
resp = s.post(login_url, headers=headers, json={"email": userdata.get('EDDMAPS_USER'), "password": userdata.get('EDDMAPS_PWD')})

# resp.raise_for_status()


In [80]:
# can't search occurrence data using scientific names directly, have to get IDs first
subj_url = "https://sandbox.bugwoodcloud.org/v2/subject"
params = dict()
params["searchon"] = "ScientificName"

for i, (k, v) in enumerate(my_vars.items()):

  namelist = []

  splist = v['sci_name'].tolist()
  splist = list(set(splist))
  # deal with alternate scientific names
  if k == "plant":
    splist.extend(["Egeria densa", "Eichhornia crassipes", "Eichhornia pontederia"])
  for name in splist:
    print(name)
    # have to drop space between genus and species
    name = name.replace(" ", "")
    params["search"] = name
    r = s.get(subj_url, params=params)

    if r.status_code == 200:
      # Parse the JSON response
      data = json.loads(r.text)
      df = json_normalize(data)
      if df.empty:
        print("No results for",name)
        continue
      else:
        # specify fields to extract
        subfields = ["subjectid", "namepart1", "namepart2"]
        df = df[subfields]
        # clean up extra taxa that contain snakehead genus name
        if k == "fish":
          mask = df[df.apply(lambda row: row.str.contains('channa', case=False).any(), axis=1)]
          if not mask.empty:
            mask = mask.loc[mask['namepart1'] != "Channa"]
            df = pd.merge(df,mask, indicator=True, how='outer') \
              .query('_merge=="left_only"') \
              .drop('_merge', axis=1)
        print (df)
        namelist.append(df)

    else:
        print("Error: ", r.status_code)

Channa
   subjectid namepart1    namepart2
0      12251    Channa        argus
1      18382    Channa     marulius
2      18674    Channa             
4      58627    Channa  micropeltes
5      58691    Channa     punctata
6      58697    Channa      bleheri
7      58698    Channa     maculata
8      58699    Channa      striata
9      58702    Channa       gachua
Proterorhinus semilunaris
   subjectid      namepart1    namepart2
0      64167  Proterorhinus  semilunaris
Neogobius melanostomus
   subjectid  namepart1     namepart2
0      12252  Neogobius  melanostomus
Osteoglossum bicirrhosum
   subjectid     namepart1    namepart2
0      20498  Osteoglossum  bicirrhosum
Clarias batrachus
   subjectid namepart1  namepart2
0      18399   Clarias  batrachus
Tinca tinca
   subjectid namepart1 namepart2
0      58693     Tinca     tinca
Misgurnus anguillicaudatus
   subjectid  namepart1         namepart2
0      18390  Misgurnus  anguillicaudatus
Osmerus mordax
   subjectid namepart1 namepart

In [79]:
mask

,0
0,True
1,True
2,True
3,True
4,True
5,True
6,True
7,True
8,True
9,True


In [ ]:

# get occurrence data
occ_url = 'https://api.bugwoodcloud.org/v2/occurrence'
params = dict()
#params["subjectid"] = "3028,12792"
# params["scientificname"] = "Hydrilla verticillata" # not a valid param, need to use subjectids
params["enddate"] = "01/17/2025"
params["startdate"] = "02/01/2024"
params["sortorder"] = "desc"
params["paging"] = "false"

r = s.get(occ_url, params=params)


In [64]:
r.json()

[{'objectid': 12344671,
  'scientificname': 'Adelges tsugae',
  'displayname': 'hemlock woolly adelgid (Adelges tsugae Annand, 1924)',
  'subjectnumber': 289,
  'habitat': 'Forest',
  'locality': None,
  'location': 'Jackson, Ohio, United States',
  'coordinates': '  39.14600, -82.70070',
  'latitude_decimal': 39.1460021,
  'longitude_decimal': -82.7007027,
  'township': None,
  'fipscode': '39079',
  'local_ownership': None,
  'sitename': None,
  'coordinateuncertaintyinmeters': 3,
  'waterbodyname': None,
  'numbercollected': None,
  'abundance': None,
  'density': None,
  'grossarea': None,
  'grossareaunits': None,
  'infestedarea': None,
  'infestedareaunits': None,
  'treatmentarea': None,
  'treatmentcomments': None,
  'disturbance': None,
  'appxquantity': 'Multiple',
  'percentcover': None,
  'eradicationdate': None,
  'plantstreated': None,
  'infestationstatus': 'Positive',
  'eradicationstatus': 'Positive',
  'eradicationstatusid': 1,
  'reportername': 'David Baker ',
  'ob

In [60]:
# Check if the request was successful
if r.status_code == 200:
  # Parse the JSON response
  data = json.loads(r.text)
  df = json_normalize(data)
  # specify fields to extract
  subfields = ["objectid", "scientificname", "observationdate", "latitude_decimal", "longitude_decimal", "recordbasis", "identificationcredibility", "verificationmethod"]
  df = df[subfields]
  print (df)

else:
    print("Error: ", r.status_code)

    objectid             scientificname           observationdate  \
0   12344671             Adelges tsugae  2025-01-16T11:48:26.000Z   
1   12344670             Adelges tsugae  2025-01-16T11:43:40.000Z   
2   12344652        Onopordum acanthium  2025-01-15T17:01:00.000Z   
3   12344649           Hyles euphorbiae  2025-01-15T16:14:00.000Z   
4   12344648         Tripidium ravennae  2025-01-15T00:00:00.000Z   
5   12344647         Tripidium ravennae  2025-01-15T00:00:00.000Z   
6   12344645               Arundo donax  2025-01-15T00:00:00.000Z   
7   12344643               Arundo donax  2025-01-15T00:00:00.000Z   
8   12344642               Arundo donax  2025-01-15T00:00:00.000Z   
9   12344639       Ageratina adenophora  2025-01-15T00:00:00.000Z   
10  12344638               Arundo donax  2025-01-15T09:58:00.000Z   
11  12344632        Reynoutria japonica  2025-01-15T00:00:00.000Z   
12  12344629          Ligustrum lucidum  2025-01-14T14:46:56.000Z   
13  12344628          Ligustrum lu

In [61]:
unique_values = df['scientificname'].unique()
print(unique_values)

['Adelges tsugae' 'Onopordum acanthium' 'Hyles euphorbiae'
 'Tripidium ravennae' 'Arundo donax' 'Ageratina adenophora'
 'Reynoutria japonica' 'Ligustrum lucidum' 'Galium verum'
 'Oenanthe javanica' 'Myosotis scorpioides' 'Digitaria ischaemum'
 'Digitaria sanguinalis' 'Sonchus asper' 'Stellaria media' 'Arctium lappa'
 'Pimpinella saxifraga' 'Eriochloa villosa' 'Cerastium fontanum'
 'Persicaria hydropiper' 'Veronica officinalis' 'Silene vulgaris'
 'Tripleurospermum inodorum' 'Senecio vulgaris' 'Plantago lanceolata'
 'Amaranthus retroflexus']


In [ ]:
identificationcredibility
verificationmethod
recordbasis
latitude_decimal
longitude_decimal

In [32]:
occ_url = "https://api.bugwoodcloud.org/v2/occurrence?page=1&pagesize=50&enddate=01/17/2025&paging=true&sortorder=desc&startdate=02/01/2024&subjectid=3028,12792"

headers = {
    "accept": "application/json",
}
resp2 = s.get(occ_url, headers=headers)


In [34]:
resp2.json()

{'nextpage': 'https://api.bugwoodcloud.org/v2/occurrence?page=2&pagesize=50&enddate=01/17/2025&paging=true&sortorder=desc&startdate=02/01/2024&subjectid=3028,12792',
 'previouspage': '',
 'page': 1,
 'totalrows': 116,
 'data': [{'objectid': 12214679,
   'scientificname': 'Hydrocharis morsus-ranae',
   'displayname': 'European frog-bit (Hydrocharis morsus-ranae L.)',
   'subjectnumber': 12792,
   'habitat': 'Aquatic: Freshwater',
   'locality': 'Small shallow back bay on Lovesick lake',
   'location': 'Peterborough, Ontario, Canada',
   'coordinates': '  44.56200, -78.22287',
   'township': None,
   'fipscode': '3515 ',
   'local_ownership': None,
   'sitename': None,
   'coordinateuncertaintyinmeters': None,
   'waterbodyname': None,
   'numbercollected': None,
   'abundance': 'Dense Monoculture',
   'density': '1-5%',
   'grossarea': 1,
   'grossareaunits': 'acres',
   'infestedarea': 100,
   'infestedareaunits': 'sq feet',
   'treatmentarea': None,
   'treatmentcomments': None,
   'd

In [36]:
resp2.url

'https://api.bugwoodcloud.org/v2/occurrence?page=1&pagesize=50&enddate=01/17/2025&paging=true&sortorder=desc&startdate=02/01/2024&subjectid=3028,12792'

In [35]:
s = requests.session()
https://api.bugwoodcloud.org
url = 'https://www.eddmaps.org/tools/index.cfm?forcelogin&'
login_data = {'username': userdata.get('EDDMAPS_USER'),
                  'password': userdata.get('EDDMAPS_PWD')}
res1 = s.post(url, login_data)

try:

    res1.raise_for_status()
except Exception as e:
    print('login failed')


login failed


In [36]:
url = 'https://www.eddmaps.org/tools/index.cfm?forcelogin&'
values = {'username': userdata.get('EDDMAPS_USER'),
          'password': userdata.get('EDDMAPS_PWD')}
r0 = requests.post(url, data=values)

In [42]:
url = 'https://sso.bugwood.org/login?app_id=eddmaps&forcelogin&'
values = {'username':'agrimm@glc.org',
          'password':'Qr7v@HDi6yZ3Nff'}
r0 = requests.post(url, data=values)

In [43]:
r0

<Response [403]>

In [ ]:

# Make a query
url2 = 'https://www.eddmaps.org/tools/query/results.cfm?reporter=&userGroupID=&observationDateStart=&observationDateEnd=&dateEnteredStart=&dateEnteredEnd=&dateUpdatedStart=&dateUpdatedEnd=&objectid=&subjectnumber=&cat=&div=&eradicationstatus=2&list=&rank=&habitat=&country=926&state=&fipscode=&township=&layersourceid=&project='
res2 = s.get(url2)
try:
    res2.raise_for_status()
except Exception as e:
    print('query failed')


Combine

In [ ]:
one dataframe - species, latlong,date,source, recordID



drop duplicates

use consistent scientific name for Brazilian waterweed, water hyacinths

save a csv for each taxon